# THEMol NSP QM Reference Instances

Downloads the **THEMol Hessian** subset from HuggingFace
(`ByteDance-Seed/THEMol`) and extracts per-instance geometry records for
N-, S-, and P-containing molecules matched to OpenFF 2.3.0 parameters.

Output is one `{param_id}_themol_instances.json` file per target parameter,
in the same `InstanceRecord` format expected by the ff_param_splitter scripts
(`analyze_parameter.py`, `validate_hierarchy.py`).

## Workflow

1. Download the CSV index (~30 MB) listing all 3.1 M Hessian molecules.
2. Filter rows whose SMILES contain N, S, or P atoms (fast string match).
3. Group matched molecules by H5 file; download only those files.
4. For each molecule: assign OpenFF parameters, measure bond/angle geometry
   from the QM-optimised coordinates (Å, degrees), build `InstanceRecord`.
5. Accumulate records per `param_id` and save as JSON.

THEMol uses **B3LYP-D3(BJ)/DZVP** level of theory — the same as the NSP
QCArchive datasets used in the existing ff_param_splitter workflow.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION — edit this cell, then run it before anything else
# ---------------------------------------------------------------------------
import os
from pathlib import Path

# ------------------------------------------------------------------
# HuggingFace cache location — SET THIS BEFORE anything else.
HF_CACHE_DIR: str | None = None  # e.g. '/scratch/your_username/hf_cache'

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')

if HF_CACHE_DIR:
    os.environ['HF_HOME'] = HF_CACHE_DIR
    os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
# ------------------------------------------------------------------

import huggingface_hub

print(f'HF_TOKEN : {"FOUND" if HF_TOKEN else "NOT FOUND — set it above manually"}')
print(f'HF cache : {HF_CACHE_DIR or "~/.cache/huggingface (default)"}')

if HF_TOKEN:
    huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
    print(f'Authenticated ({len(HF_TOKEN)} char token, huggingface_hub {huggingface_hub.__version__}).')
else:
    print('WARNING: proceeding unauthenticated — downloads will be rate-limited.')

# ------------------------------------------------------------------

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError(f'Could not locate repo root from {start}')

REPO_ROOT = _find_repo_root()

FF_NAME = 'openff-2.3.0.offxml'

OUTPUT_DIR = REPO_ROOT / 'examples' / '10_themol_qm_reference' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARTS_DIR = OUTPUT_DIR / 'parts'
PARTS_DIR.mkdir(parents=True, exist_ok=True)

INSTANCES_DIR = (
    REPO_ROOT
    / 'examples'
    / '07_single_model_benchmark'
    / 'ff_param_splitter'
    / 'instances_themol'
)
INSTANCES_DIR.mkdir(parents=True, exist_ok=True)

HF_REPO = 'ByteDance-Seed/THEMol'

# All SLURM-allocated cores. os.process_cpu_count() reads scheduler affinity,
# so this correctly reflects the SLURM allocation (not the total node count).
_allocated = os.process_cpu_count() or os.cpu_count()
N_WORKERS: int = _allocated

MAX_H5_FILES: int | None = None

# Molecules per chunk. Only this many coords + futures are in memory at once.
CHUNK_SIZE: int = 2000

# ------------------------------------------------------------------
# RESUME: H5 file stems already processed and stored in legacy.parquet.
LEGACY_H5_STEMS: list[str] = [
    # 'hessian_0', 'hessian_1', 'hessian_2', 'hessian_3',
    # 'hessian_4', 'hessian_5', 'hessian_6', 'hessian_7', 'hessian_8',
]
# ------------------------------------------------------------------

print(f'Allocated CPUs    : {_allocated}')
print(f'Worker processes  : {N_WORKERS}  (all allocated)')
print(f'Chunk size        : {CHUNK_SIZE} molecules')
print(f'Repo root         : {REPO_ROOT}')
print(f'Parts dir         : {PARTS_DIR}')
print(f'Existing parts    : {len(list(PARTS_DIR.glob("*.parquet")))}')
print(f'Legacy H5 stems   : {len(LEGACY_H5_STEMS)} declared')

In [2]:
from __future__ import annotations

import json
import sys
from collections import defaultdict
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.inchi import MolToInchi, MolToInchiKey
from tqdm.auto import tqdm

from huggingface_hub import hf_hub_download

from openff.toolkit import ForceField, Molecule, Topology

if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from mlip_optimizer.analysis.ff_param_splitter.extract import InstanceRecord, save_instances

print('Imports OK')

Imports OK


## Step 1 — Download the Hessian CSV index

The CSV is ~30 MB and lists all 3.1 M molecules with their mapped SMILES and
which H5 file they live in.  It is downloaded once and cached by
`huggingface_hub`.

In [ ]:
csv_path = hf_hub_download(
    repo_id=HF_REPO,
    filename='Hessian/hessian_dataset.csv',
    repo_type='dataset',
    cache_dir=HF_CACHE_DIR,
    token=HF_TOKEN,
)
print(f'CSV at: {csv_path}')

index_df = pd.read_csv(csv_path)
print(f'Total molecules in Hessian subset: {len(index_df):,}')
print(f'Columns: {index_df.columns.tolist()}')
print(f'Unique H5 files: {index_df["h5_file"].nunique()}')
index_df.head(3)

## Step 2 — Filter for N / S / P-containing molecules

A fast string-level pre-filter retains SMILES that contain `N`, `S`, or `P`
characters (uppercase = aliphatic; aromatic atoms `n`/`s`/`p` are also caught
by adding their lowercase forms).  No RDKit parsing needed at this stage.

In [ ]:
import gc

smiles_col = 'mapped_nonisomeric_smiles'

nsp_mask = index_df[smiles_col].str.contains('[NSPnsp]', regex=True)
nsp_df = index_df[nsp_mask].copy()

print(f'NSP molecules : {len(nsp_df):,}  ({len(nsp_df)/len(index_df):.1%} of total)')

h5_files_needed = nsp_df['h5_file'].unique()
print(f'H5 files to download: {len(h5_files_needed)}')

if MAX_H5_FILES is not None:
    h5_files_needed = h5_files_needed[:MAX_H5_FILES]
    nsp_df = nsp_df[nsp_df['h5_file'].isin(h5_files_needed)]
    print(f'(Limited to {MAX_H5_FILES} H5 files → {len(nsp_df):,} molecules)')

# Convert to plain Python before fork so workers don't inherit large DataFrames.
# With fork, Python refcounting defeats copy-on-write and every worker ends up
# with a private copy of whatever objects the parent holds.
# {h5_filename: [(uuid, cmiles), ...]}
h5_groups: dict[str, list[tuple[str, str]]] = {
    h5: list(zip(grp['uuid'], grp[smiles_col]))
    for h5, grp in nsp_df.groupby('h5_file')
}

del index_df, nsp_df
gc.collect()

print(f'Ready to process {len(h5_groups)} H5 files.')
print(f'Large DataFrames freed — worker memory footprint reduced.')

## Step 3 — Load force field (main process) and verify `_worker.py`

In [ ]:
import sys
from pathlib import Path
from openff.toolkit import ForceField

_NB_DIR = str(REPO_ROOT / 'examples' / '10_themol_qm_reference')
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)
import _worker

# Quick sanity check: run the worker init + one dummy call in the main process
ff = ForceField(FF_NAME)
print(f'ForceField loaded: {FF_NAME}')
print(f'_worker.py found : {Path(_worker.__file__).resolve()}')
print('Step 3 OK — ready for parallel processing.')

## Step 4 — Download H5 files and extract all instances

Each H5 file is downloaded once (cached by `huggingface_hub`) and processed
in parallel using **`ProcessPoolExecutor` with `spawn`**.  Each worker process
gets a fresh Python interpreter and loads its own `ForceField` once via the
initializer — this avoids the C-extension init conflicts that crash
`ThreadPoolExecutor` on cluster nodes.

Only `coords` is read per molecule; the large `hessian` arrays are skipped.

**Disk space:** ~50 H5 files × ~50–200 MB each ≈ up to ~5 GB download.
Set `MAX_H5_FILES = 1` above for a quick smoke-test.

In [ ]:
import gc
import json
import multiprocessing as mp
import sys
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, BrokenExecutor, as_completed

import pyarrow as pa
import pyarrow.parquet as pq

_NB_DIR = str(REPO_ROOT / 'examples' / '10_themol_qm_reference')
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)
import _worker  # noqa: E402

SCHEMA = pa.schema([
    ('mol_idx',    pa.int32()),
    ('inchi_key',  pa.string()),
    ('smiles',     pa.string()),
    ('cmiles',     pa.string()),
    ('param_type', pa.string()),
    ('atom_key',   pa.string()),
    ('param_id',   pa.string()),
    ('smirks',     pa.string()),
    ('qm_value',   pa.float64()),
])

_SRC_PATH = str(REPO_ROOT / 'src')
_CTX = mp.get_context('fork')

# ------------------------------------------------------------------
# Startup: clean up any .tmp files left by a previous crashed run,
# then build the set of completed H5 stems.
# ------------------------------------------------------------------

for _tmp in PARTS_DIR.glob('*.parquet.tmp'):
    _tmp.unlink()
    print(f'Removed incomplete temp file: {_tmp.name}')

_legacy = OUTPUT_DIR / 'themol_nsp_all_instances.parquet'
if _legacy.exists() and not (PARTS_DIR / 'legacy.parquet').exists():
    _legacy.rename(PARTS_DIR / 'legacy.parquet')
    print('Migrated legacy parquet → parts/legacy.parquet')
    if not LEGACY_H5_STEMS:
        print('  *** Set LEGACY_H5_STEMS in config cell to avoid re-processing. ***')

existing_parts: set[str] = {
    p.stem for p in PARTS_DIR.glob('*.parquet') if p.stem != 'legacy'
}
existing_parts.update(LEGACY_H5_STEMS)

n_remaining = sum(1 for h5 in h5_groups if Path(h5).stem not in existing_parts)
print(f'H5 files already done : {len(existing_parts)}')
print(f'H5 files remaining    : {n_remaining}')

if existing_parts or (PARTS_DIR / 'legacy.parquet').exists():
    _arrays = [
        pq.read_table(p, columns=['mol_idx'])['mol_idx']
        for p in PARTS_DIR.glob('*.parquet')
    ]
    mol_counter = (int(max(v for arr in _arrays for v in arr.to_pylist())) + 1) if _arrays else 0
    print(f'Resuming mol_counter from {mol_counter:,}')
else:
    mol_counter = 0

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def _make_pool() -> ProcessPoolExecutor:
    return ProcessPoolExecutor(
        max_workers=N_WORKERS,
        mp_context=_CTX,
        initializer=_worker.init,
        initargs=(FF_NAME, _SRC_PATH),
    )

def _collect_chunk(pool: ProcessPoolExecutor,
                   mol_data: list,
                   desc: str) -> list[dict]:
    futures = {
        pool.submit(_worker.process, cmiles, coords, mol_idx): mol_idx
        for cmiles, coords, mol_idx in mol_data
    }
    rows: list[dict] = []
    for fut in tqdm(as_completed(futures), total=len(futures), desc=desc, leave=False):
        try:
            rows.extend(fut.result())
        except Exception:
            pass
    return rows

def _emit_jsonl_from_rows(rows: list[dict], instances_dir: Path) -> int:
    """Group rows by param_id and append each group to its .jsonl file.

    Called immediately after a parquet part is atomically committed, so
    JSONL files grow incrementally with each completed H5 file — no
    separate full-dataset export pass required.
    """
    by_param: dict[str, list] = defaultdict(list)
    for row in rows:
        atom_key = [int(x) for x in row['atom_key'].split(',')]
        by_param[row['param_id']].append({
            'mol_idx':   int(row['mol_idx']),
            'inchi_key': row['inchi_key'],
            'smiles':    row['smiles'],
            'cmiles':    row['cmiles'],
            'atom_key':  atom_key,
            'qm_values': [float(row['qm_value'])],
            'qm_mean':   float(row['qm_value']),
            'mm_values': {},
            'errors':    {},
            'param_id':  row['param_id'],
            'smirks':    row['smirks'],
        })
    n = 0
    for pid, recs in by_param.items():
        path = instances_dir / f'{pid}_themol_instances.jsonl'
        with open(path, 'a') as fh:
            for rec in recs:
                fh.write(json.dumps(rec) + '\n')
        n += len(recs)
    return n

# ------------------------------------------------------------------
# Main loop — one H5 file at a time, fresh pool per file.
# Workers exit completely after each file; memory is returned to the OS
# before the next file starts.
# ------------------------------------------------------------------

n_skipped_h5 = 0
n_processed  = 0
n_instances  = 0

for h5_name, mol_pairs in tqdm(h5_groups.items(), desc='H5 files'):
    part_stem = Path(h5_name).stem
    if part_stem in existing_parts:
        n_skipped_h5 += 1
        continue

    try:
        h5_local = hf_hub_download(
            repo_id=HF_REPO,
            filename=f'Hessian/{h5_name}',
            repo_type='dataset',
            cache_dir=HF_CACHE_DIR,
            token=HF_TOKEN,
        )
    except Exception as e:
        print(f'  WARNING: could not download {h5_name}: {e}')
        continue

    with h5py.File(h5_local, 'r') as hf:
        valid_pairs = [(uuid, cmiles) for uuid, cmiles in mol_pairs if uuid in hf]

    n_mols   = len(valid_pairs)
    n_chunks = (n_mols + CHUNK_SIZE - 1) // CHUNK_SIZE

    tmp_path  = PARTS_DIR / f'{part_stem}.parquet.tmp'
    part_path = PARTS_DIR / f'{part_stem}.parquet'
    pq_writer = None
    # Accumulate all rows for this H5 file; emitted to JSONL only after the
    # parquet part is atomically committed — avoids partial/duplicate writes
    # on crash and resume.
    h5_rows: list[dict] = []

    print(f'{h5_name}: {n_mols:,} molecules, {n_chunks} chunks, {N_WORKERS} workers')

    pool = _make_pool()
    try:
        for chunk_idx in range(n_chunks):
            chunk = valid_pairs[chunk_idx * CHUNK_SIZE : (chunk_idx + 1) * CHUNK_SIZE]

            mol_data: list[tuple[str, list, int]] = []
            with h5py.File(h5_local, 'r') as hf:
                for uuid, cmiles in chunk:
                    coords = hf[uuid]['coords'][:].tolist()
                    mol_data.append((cmiles, coords, mol_counter))
                    mol_counter += 1

            desc = f'chunk {chunk_idx + 1}/{n_chunks}'
            try:
                chunk_rows = _collect_chunk(pool, mol_data, desc)
            except BrokenExecutor:
                print(f'  Pool broke on chunk {chunk_idx + 1}, recreating...')
                try:
                    pool.shutdown(wait=False, cancel_futures=True)
                except Exception:
                    pass
                pool = _make_pool()
                try:
                    chunk_rows = _collect_chunk(pool, mol_data, desc)
                except BrokenExecutor:
                    print(f'  Pool broke again — skipping chunk {chunk_idx + 1}')
                    chunk_rows = []

            del mol_data

            if chunk_rows:
                table = pa.Table.from_pylist(chunk_rows, schema=SCHEMA)
                if pq_writer is None:
                    pq_writer = pq.ParquetWriter(tmp_path, SCHEMA)
                pq_writer.write_table(table)
                n_instances += len(chunk_rows)
                h5_rows.extend(chunk_rows)  # kept for JSONL emit below

            del chunk_rows

        n_processed += n_mols

        if pq_writer is not None:
            pq_writer.close()
            pq_writer = None
            tmp_path.rename(part_path)
            existing_parts.add(part_stem)
            # Parquet committed — now safe to emit to per-param JSONL files
            n_emitted = _emit_jsonl_from_rows(h5_rows, INSTANCES_DIR)
            print(f'  -> {part_path.name} + {n_emitted:,} rows → JSONL')

    finally:
        if pq_writer is not None:
            pq_writer.close()
        pool.shutdown(wait=True, cancel_futures=True)

    del h5_rows
    gc.collect()

    try:
        h5_path = Path(h5_local)
        h5_path.resolve().unlink()
        h5_path.unlink(missing_ok=True)
    except Exception as e:
        print(f'  WARNING: could not delete {h5_name} cache: {e}')

print(f'\nDone — processed: {n_processed:,} mols | skipped: {n_skipped_h5} H5s | new instances: {n_instances:,}')
print(f'Total parts in {PARTS_DIR.name}/: {len(list(PARTS_DIR.glob("*.parquet")))}')

## Step 5 — Inspect the intermediate parquet

Run this cell at any time (including after the download is long finished) to
see which parameters are covered and how many instances exist for each.

In [ ]:
import gc
import pyarrow.dataset as ds
import pyarrow.compute as pc

# Use Arrow's columnar engine for aggregation instead of pandas.read_parquet —
# the full table is 240M+ rows with wide string columns (smiles/cmiles/smirks);
# loading that into pandas (Python objects per string) is what was OOM-killing
# this step. Only pull the narrow columns actually needed for the summary.
dataset = ds.dataset(PARTS_DIR, format='parquet')

stats_table = dataset.to_table(columns=['param_type', 'param_id', 'mol_idx', 'qm_value'])

print(f'Total rows : {stats_table.num_rows:,}')
print(f'Unique mols: {pc.count_distinct(stats_table["mol_idx"]).as_py():,}')
print(f'Parts read : {len(list(PARTS_DIR.glob("*.parquet")))}')

summary = (
    stats_table
    .group_by(['param_type', 'param_id'])
    .aggregate([
        ('qm_value', 'count'),
        ('mol_idx', 'count_distinct'),
        ('qm_value', 'min'),
        ('qm_value', 'mean'),
        ('qm_value', 'max'),
    ])
    .to_pandas()
    .rename(columns={
        'qm_value_count':          'n_instances',
        'mol_idx_count_distinct':  'n_mols',
        'qm_value_min':            'qm_min',
        'qm_value_mean':           'qm_mean',
        'qm_value_max':            'qm_max',
    })
)
summary['qm_width'] = summary['qm_max'] - summary['qm_min']
summary = summary.sort_values(['param_type', 'param_id']).reset_index(drop=True)

del stats_table
gc.collect()

summary

## Step 6 — One-time JSONL backfill from completed parquet parts

For H5 files processed **before** Step 4 was wired to emit JSONL directly,
run this cell once to populate the per-parameter `.jsonl` files from the
existing parquet parts.

Each part file (~4–5 M rows) is read one at a time and grouped by `param_id`
— the full 240 M-row table is never loaded into memory at once.  A
`.jsonl_emitted` log tracks which parts have been backfilled so the cell is
safe to re-run if interrupted.

After backfilling, **Step 4 maintains the JSONL files automatically** on every
new H5 file processed — no further sweeps are needed.

In [ ]:
import gc
import json
from collections import defaultdict

import pyarrow.parquet as pq

# Track which parts have already been backfilled — safe to re-run on interruption.
_emitted_log = PARTS_DIR / '.jsonl_emitted'
_already_emitted: set[str] = (
    set(_emitted_log.read_text().splitlines()) if _emitted_log.exists() else set()
)

_part_files = sorted(p for p in PARTS_DIR.glob('*.parquet') if p.stem != 'legacy')
_to_emit    = [p for p in _part_files if p.stem not in _already_emitted]

print(f'Parts total     : {len(_part_files)}')
print(f'Already emitted : {len(_already_emitted)}')
print(f'To backfill     : {len(_to_emit)}')

for part_path in tqdm(_to_emit, desc='Backfill JSONL'):
    # Read one part at a time — never the full 240 M-row table.
    df = pq.read_table(part_path).to_pandas()

    for pid, grp in df.groupby('param_id', sort=False):
        out_path = INSTANCES_DIR / f'{pid}_themol_instances.jsonl'
        with open(out_path, 'a') as fh:
            for row in grp.itertuples(index=False):
                atom_key = [int(x) for x in row.atom_key.split(',')]
                rec = {
                    'mol_idx':   int(row.mol_idx),
                    'inchi_key': row.inchi_key,
                    'smiles':    row.smiles,
                    'cmiles':    row.cmiles,
                    'atom_key':  atom_key,
                    'qm_values': [float(row.qm_value)],
                    'qm_mean':   float(row.qm_value),
                    'mm_values': {},
                    'errors':    {},
                    'param_id':  row.param_id,
                    'smirks':    row.smirks,
                }
                fh.write(json.dumps(rec) + '\n')

    del df
    gc.collect()

    with open(_emitted_log, 'a') as f:
        f.write(part_path.stem + '\n')

_jsonl_files = sorted(INSTANCES_DIR.glob('*_themol_instances.jsonl'))
print(f'\nBackfill done. {len(_jsonl_files)} .jsonl files in {INSTANCES_DIR.name}/')
for jf in _jsonl_files:
    n = sum(1 for _ in open(jf))
    unit = 'Å' if jf.stem.split('_')[0].startswith('b') else '°'
    print(f'  {jf.name}: {n:,} instances ({unit})')

## Using the output with ff_param_splitter

Step 4 writes one `{param_id}_themol_instances.jsonl` file per parameter
directly as each H5 file finishes — no separate export step needed.
Step 6 is a one-time backfill for parts processed before this wiring.

`load_instances` auto-detects JSON array vs JSONL format, so existing
`.json` files (QCArchive) and new `.jsonl` files (THEMol) both load
with the same call.

### Analyze a parameter

```bash
cd examples/07_single_model_benchmark/ff_param_splitter

python scripts/analyze_parameter.py \
    instances_themol/a20_themol_instances.jsonl --param a20

python scripts/analyze_parameter.py \
    instances_themol/a20_themol_instances.jsonl --param a20 --subpartition
```

### Validate a proposed hierarchy

```bash
python scripts/validate_hierarchy.py \
    instances_themol/a20_themol_instances.jsonl \
    --param a20 \
    --inline '[
        ["[*:1]~[#7X3$(*~[#6X3,#6X2,#7X2+0]):2]~[*:3]", "a20"],
        ["[*:1]-[#7X3;!r:2]-[#7X2,#7X3:3]",              "a20f"],
        ["[#6;r6:1]-[#7X3;r6:2]-[#6;r6:3]",              "a20a"]
    ]'
```

### Combine THEMol + QCArchive instances

```python
from mlip_optimizer.analysis.ff_param_splitter.extract import load_instances, save_instances

qca  = load_instances('a20_instances.json')          # JSON array — auto-detected
them = load_instances('instances_themol/a20_themol_instances.jsonl')  # JSONL — auto-detected

offset = max(r.mol_idx for r in qca) + 1
for r in them:
    r.mol_idx += offset

save_instances(qca + them, 'a20_combined_instances.json')
```

---

**Notes on THEMol vs QCArchive data:**
- Each THEMol instance has exactly **one conformer** (`qm_values` length 1);
  QCArchive instances typically have 5–10.  Both feed into `qm_mean` the same way.
- `mm_values` and `errors` are empty (no MM comparison).
- QM level: B3LYP-D3(BJ)/DZVP — identical to the NSP QCArchive sets.